# 🔍 Dossiê Exodus — Investigação de Crise de Talentos

> **Projeto 07 | MBA em Ciência de Dados — UNIFOR | Prof. Cássio Pinheiro**

---

## Contexto

Uma empresa de tecnologia com **1.200 funcionários** está vivendo uma crise de turnover: a taxa subiu de **12% para 19%** em dois anos, e cada desligamento custa **R\$ 45.000**. A VP de Pessoas pediu uma investigação profunda — e há algo mais grave escondido nos dados.

## Missões

| # | Questão investigativa |
|---|----------------------|
| 1 | Qual departamento tem o turnover mais alto? |
| 2 | Existe gap salarial de gênero? |
| 3 | Mulheres demoram mais para ser promovidas? |
| 4 | A pesquisa de clima confirma um gestor tóxico? |
| 5 | Quais fatores mais influenciam a saída? |
| 6 | Qual o custo total do turnover por departamento? |
| 7 | Como construir um scorecard de risco de saída? |

## Datasets

| Arquivo | Descrição | Registros |
|---------|-----------|----------|
| `projeto_07_people_analytics.csv` | Dados de funcionários ativos e desligados | 1.200 |
| `projeto_07_pesquisa_clima_anonima.csv` | Respostas anônimas da pesquisa de clima | 400 |
| `projeto_07_historico_promocoes.csv` | Histórico de promoções realizadas | 600 |
| `caged_consolidado.xlsx` | Dados públicos reais do CAGED/MTE | — |

---
## 📦 Módulo 0 — Importações e Configuração


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats as scipy_stats
import warnings
warnings.filterwarnings('ignore')

# ── Tema visual ──────────────────────────────────────────────
DARK_BG  = '#0A0A0B'
SURFACE  = '#111214'
BORDER   = '#252830'
TEXT     = '#E8E9EC'
MUTED    = '#5A5F6E'
RED      = '#E53935'
AMBER    = '#FFB300'
GREEN    = '#00C853'
BLUE     = '#1565C0'
PINK     = '#E91E8C'

sns.set_theme(
    style='darkgrid',
    rc={
        'figure.facecolor'     : DARK_BG,
        'axes.facecolor'       : SURFACE,
        'axes.edgecolor'       : BORDER,
        'axes.labelcolor'      : TEXT,
        'text.color'           : TEXT,
        'xtick.color'          : MUTED,
        'ytick.color'          : MUTED,
        'grid.color'           : BORDER,
        'grid.linewidth'       : 0.5,
        'legend.facecolor'     : '#1A1C20',
        'legend.edgecolor'     : BORDER,
    }
)
plt.rcParams.update({'font.family':'monospace', 'axes.titleweight':'bold', 'axes.titlecolor':TEXT})

# ── Constante de custo ───────────────────────────────────────
CUSTO_DESLIGAMENTO = 45_000

print('✅ Setup concluído.')


---
## 📂 Módulo 1 — Carregamento e Inspeção dos Dados

> Antes de qualquer análise, precisamos entender o que temos: estrutura, tipos, nulos e possíveis problemas de qualidade.


In [ ]:
# ── Carregamento ─────────────────────────────────────────────
df_people = pd.read_csv('/projeto_07_people_analytics.csv')
df_clima  = pd.read_csv('/projeto_07_pesquisa_clima_anonima.csv')
df_prom   = pd.read_csv('/projeto_07_historico_promocoes.csv')

# CAGED: tenta o caminho com barra; se falhar, tenta sem barra (mesmo diretório)
try:
    df_caged = pd.read_excel('/caged_consolidado.xlsx', sheet_name=None)
except FileNotFoundError:
    df_caged = pd.read_excel('caged_consolidado.xlsx', sheet_name=None)

print('═' * 55)
print(f'  people_analytics  : {df_people.shape[0]:>5} linhas × {df_people.shape[1]} colunas')
print(f'  pesquisa_clima    : {df_clima.shape[0]:>5} linhas × {df_clima.shape[1]} colunas')
print(f'  hist_promocoes    : {df_prom.shape[0]:>5} linhas × {df_prom.shape[1]} colunas')
print(f'  caged (abas)      : {list(df_caged.keys())}')
print('═' * 55)

### Tipos de dados, nulos e duplicatas

Uma inspeção sistemática é essencial: identificamos os problemas antes de começar a análise.


In [ ]:
def inspecionar(df, nome):
    """Retorna um resumo de tipos, nulos e estatísticas básicas."""
    resumo = pd.DataFrame({
        'dtype'        : df.dtypes,
        'nulos'        : df.isnull().sum(),
        'nulos_%'      : (df.isnull().mean() * 100).round(1),
        'únicos'       : df.nunique(),
    })
    print(f'\n{'─'*50}')
    print(f'  Dataset: {nome}  |  Duplicatas: {df.duplicated().sum()}')
    print('─'*50)
    display(resumo)

inspecionar(df_people,    'people_analytics')
inspecionar(df_clima, 'pesquisa_clima')
inspecionar(df_prom,  'historico_promocoes')


**Observações da inspeção:**
- `satisfacao_trabalho`: 45 nulos (3,75%) — preencheremos com a mediana por departamento/cargo.
- `avaliacao_desempenho`: 29 nulos (2,42%) — mesma estratégia.
- `data_saida`: 902 nulos esperados (funcionários que **não saíram** ainda).
- `comentario_aberto` em clima: 46 nulos — campo opcional, mantemos como está.
- Nenhum dataset tem duplicatas.


---
## 🧹 Módulo 2 — Limpeza e Transformação

> Tratamos nulos, corrigimos tipos, detectamos outliers e criamos as variáveis que irão alimentar a investigação.


In [ ]:
# ── 1. Converter datas ───────────────────────────────────────
df_people['data_admissao']   = pd.to_datetime(df_people['data_admissao'],   errors='coerce')
df_people['data_saida']      = pd.to_datetime(df_people['data_saida'],      errors='coerce')
df_prom['data_promocao'] = pd.to_datetime(df_prom['data_promocao'], errors='coerce')

# ── 2. Tratar nulos com mediana por grupo ─────────────────────
def impute_mediana_grupo(df, col, grupos):
    """Preenche nulos de `col` com a mediana calculada por `grupos`."""
    mediana = df.groupby(grupos)[col].transform('median')
    n_antes = df[col].isnull().sum()
    df[col]  = df[col].fillna(mediana)
    df[col]  = df[col].fillna(df[col].median())  # fallback global
    n_depois = df[col].isnull().sum()
    print(f'  [{col}] nulos: {n_antes} → {n_depois}')
    return df

print('Imputação de nulos:')
df_people = impute_mediana_grupo(df_people, 'satisfacao_trabalho',  ['departamento', 'cargo'])
df_people = impute_mediana_grupo(df_people, 'avaliacao_desempenho', ['departamento', 'cargo'])

# ── 3. Detectar e registrar outliers ─────────────────────────
def detectar_outliers_iqr(df, col):
    """Identifica outliers via IQR e retorna os índices."""
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    mask = (df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)
    return df[mask].index

print('\nOutliers detectados (IQR):')
for col in ['salario_mensal', 'horas_extras_mes', 'avaliacao_desempenho']:
    idx = detectar_outliers_iqr(df_people, col)
    print(f'  [{col}] {len(idx)} outliers — mantidos (variação legítima)')

# ── 4. Feature Engineering ───────────────────────────────────
# Score ponderado de satisfação (50% trabalho, 30% ambiente, 20% vida)
df_people['score_satisfacao'] = (
    df_people['satisfacao_trabalho']      * 0.50 +
    df_people['satisfacao_ambiente']      * 0.30 +
    df_people['equilibrio_vida_trabalho'] * 0.20
).round(3)

# Faixas de tempo na empresa (coorte)
df_people['tenure_band'] = pd.cut(
    df_people['tempo_empresa_meses'],
    bins=[0, 12, 36, 72, 9999],
    labels=['< 1 ano', '1–3 anos', '3–6 anos', '6+ anos']
)

# Desvio salarial vs mediana do cargo
med_cargo = df_people.groupby('cargo')['salario_mensal'].transform('median')
df_people['salario_vs_mediana_pct'] = ((df_people['salario_mensal'] - med_cargo) / med_cargo * 100).round(1)

# Flag de departamento crítico
df_people['dept_risco'] = df_people['departamento'].isin(['Vendas', 'Financeiro']).astype(int)

# Score de risco de saída (0–100)
def calc_score_risco(row):
    """Calcula score de risco de saída baseado nos fatores mais correlacionados."""
    s = 0
    s += 35 if row['satisfacao_trabalho'] <= 2 else 20 if row['satisfacao_trabalho'] <= 3 else 0
    s += 20 if row['promovido_ultimos_2anos'] == 0 else 0
    s += 15 if row['satisfacao_ambiente'] <= 2 else 8  if row['satisfacao_ambiente'] <= 3 else 0
    s += 15 if row['dept_risco'] == 1 else 0
    s += 10 if row['equilibrio_vida_trabalho'] <= 2 else 0
    s +=  5 if row['salario_vs_mediana_pct'] < -10 else 0
    return min(s, 100)

df_people['score_risco_saida'] = df_people.apply(calc_score_risco, axis=1)

print('\nFeatures criadas: score_satisfacao, tenure_band, salario_vs_mediana_pct, dept_risco, score_risco_saida')
display(df_people[['funcionario_id','score_satisfacao','tenure_band','salario_vs_mediana_pct','score_risco_saida']].head(4))


---
## 🔗 Módulo 3 — Cruzamento entre Datasets (Merge/Join)

> Para investigar padrões ocultos, precisamos cruzar as três fontes de dados. Usamos `merge` com estratégia `left` para preservar todos os funcionários.


In [ ]:
# ── Merge 1: People Analytics + Histórico de Promoções ───────
# Chave: funcionario_id (nem todos foram promovidos → left join)
df_people_prom = df_people.merge(
    df_prom[['funcionario_id','meses_no_cargo_anterior','aumento_percentual',
          'cargo_anterior','cargo_novo','data_promocao','avaliacao_no_momento']],
    on='funcionario_id',
    how='left'
)
print(f'PA shape original   : {df_people.shape}')
print(f'PA + Prom (left)    : {df_people_prom.shape}  → linhas extras = promoções múltiplas')
print(f'IDs com promoção    : {df_people["funcionario_id"].isin(df_prom["funcionario_id"]).sum()} de {len(df_people)}')

# ── Merge 2: People Analytics + Clima (por departamento) ─────
# Clima não tem funcionario_id → agregamos por departamento antes do merge
clima_agg = (
    df_clima.groupby('departamento')
    .agg(
        media_lideranca     = ('nota_lideranca',       'mean'),
        media_satisf_clima  = ('nota_satisfacao_geral','mean'),
        media_remuneracao   = ('nota_remuneracao',     'mean'),
        media_crescimento   = ('nota_crescimento',     'mean'),
        pct_recomendaria    = ('recomendaria_empresa', 'mean'),
    )
    .round(3)
    .reset_index()
)

df_people_full = df_people.merge(clima_agg, on='departamento', how='left')
print(f'\nPA + Clima (dept)   : {df_people_full.shape}')
print('\nAmostra do dataset consolidado:')
display(df_people_full[['funcionario_id','departamento','cargo','sexo','saiu_da_empresa',
                 'score_satisfacao','media_lideranca','score_risco_saida']].head(4))


---
## 🎯 Missão 1 — Turnover por Departamento

**Hipótese:** Existe um departamento com taxa de saída dramaticamente acima da média?  
Calculamos turnover = saídas ÷ total por departamento e comparamos com a média geral.


In [ ]:
# ── Cálculo de turnover ───────────────────────────────────────
def calcular_turnover(df, grupo):
    """Calcula taxa de turnover e custo por grupo."""
    return (
        df.groupby(grupo)
        .agg(total=('saiu_da_empresa','count'), saidas=('saiu_da_empresa','sum'))
        .assign(
            taxa_pct    = lambda x: (x['saidas'] / x['total'] * 100).round(1),
            custo_reais = lambda x: x['saidas'] * CUSTO_DESLIGAMENTO,
        )
        .sort_values('taxa_pct', ascending=False)
    )

turnover_dept = calcular_turnover(df_people, 'departamento')
taxa_media    = df_people['saiu_da_empresa'].mean() * 100

print(f'Taxa geral de turnover: {taxa_media:.1f}%')
display(turnover_dept.style
    .format({'taxa_pct':'{:.1f}%','custo_reais':'R$ {:,.0f}'})
    .background_gradient(subset='taxa_pct', cmap='RdYlGn_r')
    .set_caption('Turnover por Departamento'))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('MISSÃO 1 — TURNOVER POR DEPARTAMENTO', fontsize=14, fontweight='bold', y=1.02)

# ── VIZ 1: Barplot horizontal (taxa) ─────────────────────────
ax = axes[0]
tv  = turnover_dept.sort_values('taxa_pct')
pal = [RED if d=='Vendas' else AMBER if t>25 else BLUE
       for d, t in zip(tv.index, tv['taxa_pct'])]
bars = ax.barh(tv.index, tv['taxa_pct'], color=pal, height=0.6, alpha=0.88)
ax.axvline(taxa_media, color='white', ls='--', lw=1.5, alpha=0.7,
           label=f'Média empresa: {taxa_media:.1f}%')
for bar, val in zip(bars, tv['taxa_pct']):
    cor = RED if val > 40 else AMBER if val > 25 else BLUE
    ax.text(bar.get_width()+0.6, bar.get_y()+bar.get_height()/2,
            f'{val:.1f}%', va='center', color=cor, fontsize=10, fontweight='bold')
ax.set_xlabel('Taxa de Turnover (%)')
ax.set_title('Taxa de Saída por Departamento')
ax.legend(labelcolor='white', fontsize=9)
ax.annotate('🚨 CRÍTICO', xy=(52, 7.2), xytext=(35, 6.4),
            color=RED, fontsize=9, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=RED, lw=1.5))

# ── VIZ 2: Stacked bar (ficou vs. saiu) por departamento ──────
ax2 = axes[1]
stk = df_people.groupby('departamento')['saiu_da_empresa'].value_counts(normalize=True).unstack()
stk.columns = ['Ficou','Saiu']
stk = stk.sort_values('Saiu', ascending=True)
stk.plot(kind='barh', stacked=True, ax=ax2,
         color=[GREEN, RED], alpha=0.85, width=0.6)
ax2.set_xlabel('Proporção de Funcionários')
ax2.set_title('Proporção Ficou × Saiu\npor Departamento (stacked normalizado)')
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v*100:.0f}%'))
ax2.legend(['Ficou','Saiu'], labelcolor='white', fontsize=9)
ax2.axvline(0.5, color='white', ls=':', lw=1, alpha=0.5)

plt.tight_layout()
plt.savefig('01_turnover.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print(f'\n💡 Vendas: {turnover_dept.loc["Vendas","taxa_pct"]:.1f}% — {turnover_dept.loc["Vendas","taxa_pct"]/taxa_media:.1f}x a média.')


### Análise de Coorte Simplificada — Saída por Tempo de Empresa

Investigamos em qual fase da jornada os funcionários têm maior propensão a sair.


In [ ]:
# ── VIZ 3: Countplot com hue (tenure_band × saída) ───────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Countplot com hue
ax = axes[0]
tenure_order = ['< 1 ano', '1–3 anos', '3–6 anos', '6+ anos']
df_people_plot = df_people.copy()
df_people_plot['Resultado'] = df_people_plot['saiu_da_empresa'].map({0:'Ficou', 1:'Saiu'})
sns.countplot(
    data=df_people_plot, x='tenure_band', hue='Resultado',
    order=tenure_order, palette={'Ficou':GREEN, 'Saiu':RED},
    ax=ax, alpha=0.88
)
ax.set_xlabel('Faixa de Tempo na Empresa')
ax.set_ylabel('Número de Funcionários')
ax.set_title('Contagem Ficou × Saiu\npor Tempo de Empresa (Countplot)')
ax.legend(title='Resultado', labelcolor='white', fontsize=9)

# Taxa de saída por tenure
ax2 = axes[1]
taxa_tenure = df_people.groupby('tenure_band')['saiu_da_empresa'].mean() * 100
taxa_tenure = taxa_tenure.reindex(tenure_order)
cores_t = [RED if v > taxa_media else AMBER if v > taxa_media*0.9 else GREEN for v in taxa_tenure]
bars = ax2.bar(taxa_tenure.index, taxa_tenure.values, color=cores_t, alpha=0.88, width=0.6)
ax2.axhline(taxa_media, color='white', ls='--', lw=1.5, alpha=0.7, label=f'Média: {taxa_media:.1f}%')
for bar, val in zip(bars, taxa_tenure.values):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold', color='white')
ax2.set_xlabel('Faixa de Tempo na Empresa')
ax2.set_ylabel('Taxa de Saída (%)')
ax2.set_title('Taxa de Saída por Coorte de Tenure\n(Análise de Cohort Simplificada)')
ax2.legend(labelcolor='white', fontsize=9)

plt.tight_layout()
plt.savefig('02_coorte_tenure.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()


---
## 🎯 Missão 2 — Gap Salarial de Gênero

**Hipótese:** Mulheres recebem sistematicamente menos que homens no mesmo cargo e departamento.  
**Metodologia:** `groupby` com múltiplas agregações + cálculo de proporções e gap percentual.


In [ ]:
# ── Análise bivariada: salário por cargo × sexo ───────────────
gap_cargo = (
    df_people.groupby(['cargo','sexo'])['salario_mensal']
    .agg(['mean','median','std','count'])
    .round(2)
)
display(gap_cargo)

# Pivot para calcular gap
gap_pivot = df_people.groupby(['cargo','sexo'])['salario_mensal'].mean().unstack()
gap_pivot['gap_pct']   = (gap_pivot['M'] - gap_pivot['F']) / gap_pivot['F'] * 100
gap_pivot['gap_reais'] =  gap_pivot['M'] - gap_pivot['F']

ordem_cargo = ['Júnior','Pleno','Sênior','Coordenador','Gerente']
gap_pivot = gap_pivot.reindex(ordem_cargo)

print('\nGap salarial por cargo (M recebe X% a mais que F):')
display(gap_pivot.style
    .format({'F':'R$ {:,.0f}','M':'R$ {:,.0f}','gap_pct':'+{:.1f}%','gap_reais':'R$ {:,.0f}'})
    .background_gradient(subset='gap_pct', cmap='Oranges'))

# Por departamento
gap_dept = (
    df_people.groupby(['departamento','sexo'])['salario_mensal'].mean().unstack()
    .assign(gap_pct=lambda x: (x['M']-x['F'])/x['F']*100)
    .sort_values('gap_pct', ascending=False)
)
print('\nGap salarial por departamento:')
display(gap_dept.round(2).style.background_gradient(subset='gap_pct', cmap='Oranges'))

gap_medio = gap_pivot['gap_pct'].mean()
print(f'\n📊 Gap médio geral: {gap_medio:.1f}% | Maior: Gerente {gap_pivot.loc["Gerente","gap_pct"]:.1f}%')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('MISSÃO 2 — GAP SALARIAL DE GÊNERO', fontsize=14, fontweight='bold', y=1.02)

# ── VIZ 4: Boxplot comparativo M vs F por cargo ───────────────
ax = axes[0]
sns.boxplot(
    data=df_people, x='cargo', y='salario_mensal', hue='sexo',
    order=ordem_cargo, palette={'F':PINK, 'M':BLUE},
    ax=ax, width=0.6, linewidth=1.2
)
ax.set_xlabel('Cargo')
ax.set_ylabel('Salário Mensal (R$)')
ax.set_title('Distribuição Salarial por Cargo\ne Gênero (Boxplot Comparativo)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'R${v/1000:.0f}k'))
ax.set_xticklabels(ax.get_xticklabels(), rotation=12)
ax.legend(title='Sexo', labelcolor='white', fontsize=9)

# ── VIZ 5: Catplot — salário médio por dept e sexo ────────────
ax2 = axes[1]
dept_sal = df_people.groupby(['departamento','sexo'])['salario_mensal'].mean().reset_index()
dept_pivot = dept_sal.pivot(index='departamento', columns='sexo', values='salario_mensal').sort_values('M')
x = np.arange(len(dept_pivot))
w = 0.38
ax2.barh(x-w/2, dept_pivot['F'], w, label='Mulheres', color=PINK, alpha=0.85)
ax2.barh(x+w/2, dept_pivot['M'], w, label='Homens',   color=BLUE, alpha=0.85)
ax2.set_yticks(x); ax2.set_yticklabels(dept_pivot.index, fontsize=9)
ax2.set_xlabel('Salário Médio (R$)')
ax2.set_title('Salário Médio por Departamento\ne Gênero')
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'R${v/1000:.0f}k'))
ax2.legend(title='Gênero', labelcolor='white', fontsize=9)

# ── VIZ 6: Gap % por cargo (bar) ──────────────────────────────
ax3 = axes[2]
cores_gap = [RED if g > 20 else AMBER for g in gap_pivot['gap_pct']]
bars = ax3.bar(gap_pivot.index, gap_pivot['gap_pct'], color=cores_gap, alpha=0.88, width=0.55)
ax3.axhline(gap_pivot['gap_pct'].mean(), color='white', ls='--', lw=1.5, alpha=0.6,
            label=f'Média: {gap_pivot["gap_pct"].mean():.1f}%')
for bar, val in zip(bars, gap_pivot['gap_pct']):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
             f'+{val:.1f}%', ha='center', fontsize=11, fontweight='bold', color='white')
ax3.set_xlabel('Cargo'); ax3.set_ylabel('Gap (Homens ganham X% a mais)')
ax3.set_title('Gap Salarial (%) por Cargo\n(Homens vs. Mulheres)')
ax3.set_ylim(0, 26); ax3.legend(labelcolor='white', fontsize=9)

plt.tight_layout()
plt.savefig('03_gap_salarial.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()


---
## 🎯 Missão 3 — Mulheres Demoram Mais para ser Promovidas?

**Hipótese:** O tempo de espera para promoção é maior para mulheres.  
Analisamos `meses_no_cargo_anterior` no histórico de promoções + teste estatístico.


In [ ]:
# ── Groupby com múltiplas agregações ─────────────────────────
tempo_sexo = df_prom.groupby('sexo')['meses_no_cargo_anterior'].agg(
    media='mean', mediana='median', desvio='std', n='count'
).round(2)
display(tempo_sexo)

dif_abs = tempo_sexo.loc['F','media'] - tempo_sexo.loc['M','media']
dif_pct = dif_abs / tempo_sexo.loc['M','media'] * 100
print(f'Mulheres esperam {dif_abs:.1f} meses a mais para ser promovidas (+{dif_pct:.1f}%)')

# ── Teste de Mann-Whitney ─────────────────────────────────────
f_vals = df_prom[df_prom['sexo']=='F']['meses_no_cargo_anterior']
m_vals = df_prom[df_prom['sexo']=='M']['meses_no_cargo_anterior']
stat, pval = scipy_stats.mannwhitneyu(f_vals, m_vals, alternative='greater')
print(f'\nMann-Whitney U={stat:.0f}, p={pval:.4f}')
print(f'{"✅ Diferença SIGNIFICATIVA (p<0.05)" if pval<0.05 else "❌ Não significativo"}')

# Tabela por cargo + sexo
print('\nTempo médio por cargo de origem:')
display(
    df_prom.groupby(['cargo_anterior','sexo'])['meses_no_cargo_anterior']
    .mean().round(1).unstack()
    .assign(gap_meses=lambda x: (x['F']-x['M']).round(1))
)

print('\nAumento % médio na promoção por sexo:')
display(df_prom.groupby('sexo')['aumento_percentual'].agg(['mean','median']).round(2))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('MISSÃO 3 — ACESSO A PROMOÇÕES POR GÊNERO', fontsize=14, fontweight='bold', y=1.02)

# ── VIZ 7: Boxplot comparativo M vs F (tempo até promoção) ────
ax = axes[0]
bp = ax.boxplot([f_vals, m_vals], labels=['Mulheres','Homens'],
               patch_artist=True, widths=0.5,
               medianprops=dict(color='white', linewidth=2.5))
bp['boxes'][0].set(facecolor=PINK, alpha=0.72)
bp['boxes'][1].set(facecolor=BLUE, alpha=0.72)
for w in bp['whiskers']+bp['caps']+bp['fliers']: w.set_color(MUTED)
ax.set_ylabel('Meses no cargo anterior')
ax.set_title(f'Tempo até Promoção por Gênero\n(Mann-Whitney p={pval:.4f} ✅)')
ax.text(1, f_vals.mean()+0.5, f'média\n{f_vals.mean():.1f}m', ha='center', color=PINK, fontsize=9, fontweight='bold')
ax.text(2, m_vals.mean()+0.5, f'média\n{m_vals.mean():.1f}m', ha='center', color=BLUE, fontsize=9, fontweight='bold')

# ── VIZ 8: Violinplot — distribuição de tempo por sexo ────────
ax2 = axes[1]
sns.violinplot(
    data=df_prom, x='sexo', y='meses_no_cargo_anterior',
    palette={'F':PINK, 'M':BLUE}, ax=ax2,
    inner='quartile', alpha=0.8
)
ax2.set_xlabel('Gênero')
ax2.set_ylabel('Meses no cargo anterior')
ax2.set_title('Distribuição do Tempo até Promoção\n(Violinplot com quartis internos)')
ax2.set_xticklabels(['Mulheres','Homens'])

# ── VIZ 9: Catplot — tempo por cargo e sexo ───────────────────
ax3 = axes[2]
tc = df_prom.groupby(['cargo_anterior','sexo'])['meses_no_cargo_anterior'].mean().unstack()
ordem_c = [c for c in ['Júnior','Pleno','Sênior','Coordenador'] if c in tc.index]
tc = tc.reindex(ordem_c)
x3 = np.arange(len(tc))
ax3.bar(x3-0.2, tc['F'], 0.38, label='Mulheres', color=PINK, alpha=0.88)
ax3.bar(x3+0.2, tc['M'], 0.38, label='Homens',   color=BLUE, alpha=0.88)
for i, (f, m) in enumerate(zip(tc['F'], tc['M'])):
    ax3.text(i-0.2, f+0.3, f'{f:.0f}m', ha='center', fontsize=8, color=PINK)
    ax3.text(i+0.2, m+0.3, f'{m:.0f}m', ha='center', fontsize=8, color=BLUE)
ax3.set_xticks(x3); ax3.set_xticklabels(tc.index, rotation=10)
ax3.set_ylabel('Meses médios no cargo anterior')
ax3.set_title('Tempo até Promoção\npor Cargo de Origem')
ax3.legend(labelcolor='white', fontsize=9)

plt.tight_layout()
plt.savefig('04_promocoes.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()


---
## 🎯 Missão 4 — Pesquisa de Clima: O Gestor Tóxico

**Hipótese:** Existe um departamento com nota de liderança anormalmente baixa, e os comentários abertos revelam padrões de assédio.  
**Técnicas:** Heatmap de correlação entre notas + análise textual dos comentários + merge com dados de turnover.


In [ ]:
cols_notas = ['nota_satisfacao_geral','nota_lideranca','nota_remuneracao','nota_crescimento']

# ── Groupby com múltiplas agregações ─────────────────────────
media_clima = (
    df_clima.groupby('departamento')[cols_notas + ['recomendaria_empresa']]
    .agg(['mean','std'])
    .round(2)
)
display(media_clima)

# Versão simples para plots
clima_simple = df_clima.groupby('departamento')[cols_notas].mean().round(2)
clima_simple['recomendaria_pct'] = df_clima.groupby('departamento')['recomendaria_empresa'].mean().round(3)
clima_simple = clima_simple.sort_values('nota_lideranca')

# ── Merge df_clima com turnover (cruzamento entre datasets) ──────
tv_pct = turnover_dept['taxa_pct'].rename('turnover_pct')
clima_merged = clima_simple.join(tv_pct)
print('\nClima + Turnover (merge por departamento):')
display(clima_merged)

# Correlação entre nota de liderança e turnover
r, p = scipy_stats.pearsonr(clima_merged['nota_lideranca'], clima_merged['turnover_pct'])
print(f'\nCorrelação liderança × turnover: r={r:.3f}, p={p:.4f}')
print(f'→ {"Correlação NEGATIVA significativa" if p < 0.1 else "Tendência negativa"}: piores notas de liderança → maior turnover')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 6))
fig.suptitle('MISSÃO 4 — PESQUISA DE CLIMA ORGANIZACIONAL', fontsize=14, fontweight='bold', y=1.02)

# ── VIZ 10: Heatmap das notas de df_clima ───────────────────────
ax = axes[0]
cmap_rg = LinearSegmentedColormap.from_list('rg', [RED, AMBER, GREEN])
im = ax.imshow(clima_simple[cols_notas].values, cmap=cmap_rg, vmin=1, vmax=5, aspect='auto')
ax.set_xticks(range(4))
ax.set_xticklabels(['Satisf.','Liderança','Remuner.','Crescimento'], fontsize=9)
ax.set_yticks(range(len(clima_simple)))
ax.set_yticklabels(clima_simple.index, fontsize=10)
for i in range(len(clima_simple)):
    for j, c in enumerate(cols_notas):
        val = clima_simple[c].iloc[i]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9,
                fontweight='bold', color='black' if val > 2.8 else 'white')
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Nota (1–5)', color=TEXT)
ax.set_title('Heatmap de Notas de Clima\npor Departamento (1=péssimo, 5=excelente)')

# ── VIZ 11: Scatter liderança × turnover ──────────────────────
ax2 = axes[1]
for dept, row in clima_merged.iterrows():
    cor = RED if dept == 'Vendas' else AMBER if row['turnover_pct'] > 25 else BLUE
    ax2.scatter(row['nota_lideranca'], row['turnover_pct'], s=120, color=cor, zorder=5)
    ax2.annotate(dept, (row['nota_lideranca'], row['turnover_pct']),
                 textcoords='offset points', xytext=(6, 3), fontsize=8, color=TEXT)
m, b = np.polyfit(clima_merged['nota_lideranca'], clima_merged['turnover_pct'], 1)
xr = np.linspace(clima_merged['nota_lideranca'].min(), clima_merged['nota_lideranca'].max(), 50)
ax2.plot(xr, m*xr+b, color='white', ls='--', lw=1.5, alpha=0.5, label=f'r={r:.2f}')
ax2.set_xlabel('Nota Média de Liderança (1–5)')
ax2.set_ylabel('Taxa de Turnover (%)')
ax2.set_title('Liderança × Turnover\n(Cruzamento Clima + People Analytics)')
ax2.legend(labelcolor='white', fontsize=9)

# ── VIZ 12: Bar % recomendaria ────────────────────────────────
ax3 = axes[2]
rec = (clima_simple['recomendaria_pct'] * 100).sort_values()
cores_rec = [RED if v < 30 else AMBER if v < 55 else GREEN for v in rec]
bars3 = ax3.barh(rec.index, rec.values, color=cores_rec, height=0.6, alpha=0.88)
ax3.axvline(50, color='white', ls='--', lw=1.5, alpha=0.5, label='50%')
for bar, val in zip(bars3, rec.values):
    cor = RED if val < 30 else AMBER if val < 55 else GREEN
    ax3.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
             f'{val:.1f}%', va='center', color=cor, fontsize=10, fontweight='bold')
ax3.set_xlabel('% que Recomendaria a Empresa')
ax3.set_title('"Você recomendaria esta empresa\ncomo bom lugar para trabalhar?"')
ax3.legend(labelcolor='white', fontsize=9)
ax3.set_xlim(0, 85)

plt.tight_layout()
plt.savefig('05_clima.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()


### Análise Textual — Comentários Anônimos

Extraímos e classificamos as palavras-chave de alerta nos comentários do departamento mais crítico.


In [ ]:
def analisar_comentarios(df_clima, departamento):
    """Extrai e categoriza comentários de um departamento."""
    comentarios = df_clima[df_clima['departamento'] == departamento]['comentario_aberto'].dropna()
    unicos = comentarios.unique()

    keywords = {
        '🚨 Assédio moral'  : ['grita','humilha','medo','pressão absurda','abuso'],
        '🚨 Discriminação'  : ['machista','mulher','promovido','diferença'],
        '⚠️  Liderança'     : ['gestor','chefe','ninguém','culpam'],
        '⚠️  Reconhecimento': ['reconhecimento','impossível','resultado'],
    }

    texto_total = ' '.join(comentarios.str.lower())
    print(f'Departamento: {departamento.upper()} | {len(comentarios)} comentários | {len(unicos)} únicos')
    print('─'*60)
    for i, c in enumerate(unicos, 1):
        print(f'[{i:02d}] "{c}"')

    print('\nPALAVRAS-CHAVE DETECTADAS:')
    for categoria, kws in keywords.items():
        hits = [kw for kw in kws if kw in texto_total]
        if hits:
            print(f'  {categoria}: {hits}')

pior_dept = clima_simple.index[0]  # menor nota de liderança
analisar_comentarios(df_clima, pior_dept)


---
## 🎯 Missão 5 — Fatores que Influenciam a Saída (Multivariado)

**Objetivo:** Identificar quais variáveis diferenciam quem sai de quem fica, usando heatmap de correlação, análise bivariada e cruzamento com o dataset de clima.


In [ ]:
num_cols = [
    'satisfacao_trabalho','satisfacao_ambiente','equilibrio_vida_trabalho',
    'salario_mensal','horas_extras_mes','avaliacao_desempenho',
    'promovido_ultimos_2anos','idade','tempo_empresa_meses',
    'num_projetos','score_satisfacao','score_risco_saida'
]

# ── Heatmap de correlação ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('MISSÃO 5 — HEATMAP DE CORRELAÇÃO E ANÁLISE MULTIVARIADA', fontsize=14, fontweight='bold', y=1.02)

# VIZ 13: Heatmap de correlação completo
ax = axes[0]
corr_matrix = df_people[num_cols + ['saiu_da_empresa']].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap_div = LinearSegmentedColormap.from_list('div', [RED,'#1A1C20',BLUE])
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
labels_short = ['Sat.Trab','Sat.Amb','Equilíbrio','Salário','Hrs.Extra',
                'Avaliação','Promovido','Idade','Tempo Emp.','Nº Proj.',
                'Score Sat.','Score Risco','Saiu']
ax.set_xticks(range(len(corr_matrix))); ax.set_xticklabels(labels_short, rotation=70, ha='right', fontsize=7)
ax.set_yticks(range(len(corr_matrix))); ax.set_yticklabels(labels_short, fontsize=7)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Heatmap de Correlação\nEntre Todas as Variáveis Numéricas')

# VIZ 14: Correlação com 'saiu_da_empresa'
ax2 = axes[1]
corr_saida = corr_matrix['saiu_da_empresa'].drop('saiu_da_empresa').sort_values()
label_map = {
    'satisfacao_trabalho':'Satisf. Trabalho','satisfacao_ambiente':'Satisf. Ambiente',
    'equilibrio_vida_trabalho':'Equilíbrio Vida','salario_mensal':'Salário',
    'horas_extras_mes':'Horas Extras','avaliacao_desempenho':'Avaliação',
    'promovido_ultimos_2anos':'Promovido','idade':'Idade',
    'tempo_empresa_meses':'Tempo Empresa','num_projetos':'Nº Projetos',
    'score_satisfacao':'Score Satisfação [FE]','score_risco_saida':'Score Risco [FE]'
}
labels = [label_map.get(c,c) for c in corr_saida.index]
cores_c = [RED if v < -0.15 else AMBER if v < 0 else GREEN if v > 0.05 else MUTED for v in corr_saida]
bars = ax2.barh(labels, corr_saida.values, color=cores_c, height=0.65, alpha=0.88)
ax2.axvline(0, color='white', lw=1, alpha=0.5)
for bar, val in zip(bars, corr_saida.values):
    ax2.text(val + (0.004 if val>=0 else -0.004), bar.get_y()+bar.get_height()/2,
             f'{val:+.3f}', va='center', ha='left' if val>=0 else 'right', fontsize=9)
ax2.set_xlabel('Correlação de Pearson com Saída')
ax2.set_title('Correlação com Saída da Empresa\n[FE] = Feature Engineered')

plt.tight_layout()
plt.savefig('06_correlacao.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()


In [ ]:
# ── Perfil detalhado: ficou vs. saiu ─────────────────────────
perfil = df_people.groupby('saiu_da_empresa')[num_cols].mean().T
perfil.columns = ['Ficou','Saiu']
perfil['Δ %'] = (perfil['Saiu'] - perfil['Ficou']) / perfil['Ficou'] * 100
display(perfil.round(3).style.background_gradient(subset='Δ %', cmap='RdYlGn_r'))

# ── Cruzamento com clima: satisfação e nota de liderança ──────
print('\nTurnover alto correlaciona com pior liderança (merge Missão 1 + 4):')
display(clima_merged[['nota_lideranca','nota_satisfacao_geral','turnover_pct']].sort_values('turnover_pct', ascending=False))

# ── Análise por faixa de satisfação ──────────────────────────
faixas_sat = df_people.groupby(pd.cut(df_people['satisfacao_trabalho'], bins=[0,1,2,3,4,5]))['saiu_da_empresa'].mean()*100
print('\nTaxa de saída por faixa de satisfação:')
print(faixas_sat.round(1))


---
## 🎯 Missão 6 — Custo do Turnover por Departamento

Estimativa do custo financeiro dos desligamentos nos últimos 12 meses.


In [ ]:
DATA_CORTE = pd.Timestamp('2024-01-01')

saidas_12m = df_people[(df_people['saiu_da_empresa'] == 1) & (df_people['data_saida'] >= DATA_CORTE)]

custo_dept = (
    saidas_12m.groupby('departamento')
    .agg(saidas=('saiu_da_empresa','count'), salario_medio=('salario_mensal','mean'))
    .assign(
        custo_direto = lambda x: x['saidas'] * CUSTO_DESLIGAMENTO,
        pct_total    = lambda x: x['saidas'] / x['saidas'].sum() * 100,
    )
    .sort_values('custo_direto', ascending=False)
)

display(custo_dept.style
    .format({'salario_medio':'R$ {:,.0f}','custo_direto':'R$ {:,.0f}','pct_total':'{:.1f}%'})
    .background_gradient(subset='custo_direto', cmap='Reds'))

custo_total = custo_dept['custo_direto'].sum()
print(f'\n💰 CUSTO TOTAL: R$ {custo_total:,.0f}')

# ROI da retenção
n_atual  = df_people['saiu_da_empresa'].sum()
n_ideal  = round(len(df_people) * 0.15)  # benchmark 15% setor tech
economia = (n_atual - n_ideal) * CUSTO_DESLIGAMENTO
print(f'\nSe reduzirmos o turnover para 15% (benchmark setor tech BR):')
print(f'  Desligamentos a menos: {n_atual - n_ideal}')
print(f'  Economia potencial   : R$ {economia:,.0f}/ano')


---
## 🎯 Missão 7 — Scorecard de Risco de Saída

**Objetivo:** Identificar proativamente funcionários em risco de desligamento para ação preventiva de RH.

| Faixa | Nível | Ação |
|-------|-------|------|
| 0–40 | 🟢 Baixo | Monitoramento padrão |
| 41–70 | 🟡 Médio | Conversa 1:1 em 30 dias |
| 71–100 | 🔴 Alto | Intervenção imediata |

In [ ]:
def classifica_risco(s):
    """Classifica o score de risco em categoria."""
    if s <= 40: return '🟢 Baixo'
    elif s <= 70: return '🟡 Médio'
    return '🔴 Alto'

df_people['risco_class'] = df_people['score_risco_saida'].apply(classifica_risco)

# Distribuição
print('Distribuição por categoria de risco:')
display(df_people['risco_class'].value_counts().to_frame())

# Validação: taxa real de saída por categoria
print('\nValidação — taxa real de saída por categoria:')
display(df_people.groupby('risco_class')['saiu_da_empresa'].mean().mul(100).round(1).to_frame('Taxa saída real (%)').sort_values('Taxa saída real (%)', ascending=False))

# Top funcionários ativos em risco alto
alto_risco = df_people[(df_people['saiu_da_empresa']==0) & (df_people['score_risco_saida'] > 70)]
print(f'\nFuncionários ATIVOS em alto risco: {len(alto_risco)}')
print(f'Custo potencial se saírem todos  : R$ {len(alto_risco)*CUSTO_DESLIGAMENTO:,.0f}')
print('\nTop 10 casos prioritários:')
display(alto_risco[['funcionario_id','departamento','cargo','sexo','score_risco_saida',
                    'satisfacao_trabalho','promovido_ultimos_2anos']]
        .sort_values('score_risco_saida', ascending=False).head(10))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('MISSÃO 7 — SCORECARD DE RISCO DE SAÍDA', fontsize=14, fontweight='bold', y=1.02)

# VIZ 15: Histograma — score por resultado real
ax = axes[0]
bins = range(0, 105, 5)
ax.hist(df_people[df_people['saiu_da_empresa']==0]['score_risco_saida'], bins=bins, color=GREEN, alpha=0.65, label='Ficou', density=True)
ax.hist(df_people[df_people['saiu_da_empresa']==1]['score_risco_saida'], bins=bins, color=RED,   alpha=0.65, label='Saiu',  density=True)
ax.axvline(40, color='yellow', ls='--', lw=1.5, alpha=0.8)
ax.axvline(70, color=AMBER,   ls='--', lw=1.5, alpha=0.8)
ax.set_xlabel('Score de Risco (0–100)'); ax.set_ylabel('Densidade')
ax.set_title('Score de Risco por Resultado Real')
ax.legend(labelcolor='white', fontsize=9)

# VIZ 16: Donut — distribuição por categoria
ax2 = axes[1]
cat_order = ['🟢 Baixo','🟡 Médio','🔴 Alto']
cat_count = df_people['risco_class'].value_counts().reindex([c for c in cat_order if c in df_people['risco_class'].unique()])
wedges, texts, autotexts = ax2.pie(
    cat_count.values, labels=cat_count.index, colors=[GREEN, AMBER, RED],
    autopct='%1.1f%%', startangle=90, pctdistance=0.75,
    wedgeprops=dict(edgecolor=DARK_BG, linewidth=2.5)
)
for t in texts+autotexts: t.set_color('white'); t.set_fontsize(10)
ax2.set_title('Distribuição por\nCategoria de Risco')

# VIZ 17: Score médio por departamento (ativos)
ax3 = axes[2]
sc_dept = df_people[df_people['saiu_da_empresa']==0].groupby('departamento')['score_risco_saida'].mean().sort_values(ascending=False)
cores_sc = [RED if v>55 else AMBER if v>40 else GREEN for v in sc_dept]
bars3 = ax3.barh(sc_dept.index[::-1], sc_dept.values[::-1], color=cores_sc[::-1], height=0.6, alpha=0.88)
for bar, val in zip(bars3, sc_dept.values[::-1]):
    ax3.text(bar.get_width()+0.4, bar.get_y()+bar.get_height()/2,
             f'{val:.1f}', va='center', fontsize=10)
ax3.set_xlabel('Score Médio de Risco')
ax3.set_title('Score Médio por Departamento\n(Funcionários Ativos)')

plt.tight_layout()
plt.savefig('07_scorecard.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()


---
## 🌟 Bônus — Enriquecimento com Dados Públicos Reais (CAGED/MTE)

> **Fonte:** CAGED — Cadastro Geral de Empregados e Desempregados, Ministério do Trabalho e Emprego.  
> URL: https://bi.mte.gov.br/bgcaged/

Usamos o CAGED para responder:
1. O gap salarial de gênero desta empresa está acima ou abaixo da média nacional?
2. A taxa de turnover da empresa é realista para o setor de TI no Brasil?
3. Como evolui o salário médio do setor de Informação/Comunicação?


In [ ]:
# ── Verificar abas disponíveis ───────────────────────────────
print('Abas disponíveis no CAGED:', list(df_caged.keys()))

# Nomes reais das abas (detecção automática)
aba_perfil = next((k for k in df_caged if 'Perfil' in k), None)
aba_cnae   = next((k for k in df_caged if 'CNAE' in k or 'Setor' in k), None)

if aba_perfil is None or aba_cnae is None:
    print('\n⚠️  Abas esperadas não encontradas.')
    print(f'   Aba Perfil : {aba_perfil}')
    print(f'   Aba CNAE   : {aba_cnae}')
    print('   Verifique os nomes das abas acima e ajuste se necessário.')
else:
    df_perfil = df_caged[aba_perfil].copy()
    df_perfil.columns = df_perfil.columns.str.strip()

    # Gap salarial nacional (CAGED 2022–2025)
    gap_nac     = df_perfil[df_perfil['Ano'] >= 2022].groupby('Sexo')['Salario Medio (R$)'].mean()
    gap_nac_pct = (gap_nac['Masculino'] - gap_nac['Feminino']) / gap_nac['Feminino'] * 100

    # gap_empresa: recalculado aqui para não depender da Missão 2 ter rodado
    _gap = df_people.groupby(['cargo', 'sexo'])['salario_mensal'].mean().unstack()
    _gap['gap_pct'] = (_gap['M'] - _gap['F']) / _gap['F'] * 100
    gap_empresa = _gap['gap_pct'].mean()

    print('\nGAP SALARIAL — Empresa vs. Brasil (CAGED 2022–2025)')
    print(f'  Nacional  : {gap_nac_pct:.1f}% (M sobre F)')
    print(f'  Esta empr.: {gap_empresa:.1f}% (M sobre F)')
    print(f'  Diferença : {gap_empresa - gap_nac_pct:+.1f}pp')

    # Evolução salarial TI
    df_cnae   = df_caged[aba_cnae].copy()
    df_cnae.columns = df_cnae.columns.str.strip()
    col_setor = next((c for c in df_cnae.columns if 'Setor' in c or 'setor' in c), None)
    val_ti    = next((v for v in df_cnae[col_setor].unique()
                      if 'Informa' in str(v) or 'Comuni' in str(v)), None)
    print(f'\nSetor TI identificado: {val_ti!r}')

    ti          = df_cnae[df_cnae[col_setor] == val_ti].groupby('Ano')['Salario Medio (R$)'].mean()
    sal_empresa = df_people['salario_mensal'].mean()

    print(f'Salário médio TI Brasil (mais recente): R$ {ti.iloc[-1]:,.0f}')
    print(f'Salário médio desta empresa            : R$ {sal_empresa:,.0f}')
    print(f'Empresa paga {sal_empresa / ti.iloc[-1]:.1f}x a média nacional do setor')
    print('\n→ O problema de retenção NÃO é salarial: a empresa paga acima da média.')
    print('  É comportamental: liderança tóxica e ausência de reconhecimento.')


In [ ]:
if aba_perfil and aba_cnae:
    # recalcula localmente para não depender da Missão 1 ter rodado
    taxa_media_local = df_people['saiu_da_empresa'].mean() * 100

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle('BONUS — CONTEXTUALIZACAO COM DADOS REAIS CAGED/MTE',
                 fontsize=13, fontweight='bold', color=AMBER, y=1.02)

    # Gráfico 1: Gap empresa vs. nacional
    ax = axes[0]
    vals = [gap_nac_pct, gap_empresa]
    bars = ax.bar(['Gap Nacional\n(CAGED)', 'Gap desta\nEmpresa'], vals,
                  color=[BLUE, RED if gap_empresa > gap_nac_pct else AMBER], alpha=0.88, width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                f'{val:.1f}%', ha='center', fontsize=14, fontweight='bold', color='white')
    ax.set_ylabel('Gap M->F (%)'); ax.set_title('Gap Salarial de Genero:\nEmpresa vs. Media Nacional')
    ax.set_ylim(0, 24)
    ax.text(0.5, 0.06, 'Fonte: CAGED/MTE 2022-2025', transform=ax.transAxes,
            ha='center', fontsize=8, color=MUTED, style='italic')

    # Gráfico 2: Evolução salarial TI Brasil
    ax2 = axes[1]
    ax2.plot(ti.index, ti.values, color=BLUE, marker='o', lw=2.5, label='CAGED — TI BR')
    ax2.axhline(sal_empresa, color=AMBER, ls='--', lw=2,
                label=f'Esta empresa: R$ {sal_empresa:,.0f}')
    ax2.fill_between(ti.index, ti.values, alpha=0.1, color=BLUE)
    for x, y in zip(ti.index, ti.values):
        ax2.text(x, y+30, f'R${y/1000:.1f}k', ha='center', fontsize=8, color=BLUE)
    ax2.set_xlabel('Ano'); ax2.set_ylabel('Salario Medio (R$)')
    ax2.set_title('Evolucao Salarial\nSetor TI/Comunicacao Brasil')
    ax2.legend(labelcolor='white', fontsize=9)
    ax2.text(0.5, 0.04, 'Fonte: CAGED/MTE', transform=ax2.transAxes,
             ha='center', fontsize=8, color=MUTED, style='italic')

    # Gráfico 3: Turnover vs. benchmark
    ax3 = axes[2]
    bench_l = ['Mercado BR\n(ref.)', 'Setor Tech\nBR (tipico)', 'Esta\nEmpresa']
    bench_v = [15.0, 18.0, taxa_media_local]
    bench_c = [GREEN, AMBER, RED if taxa_media_local > 18 else AMBER]
    bars3 = ax3.bar(bench_l, bench_v, color=bench_c, alpha=0.88, width=0.55)
    for bar, val in zip(bars3, bench_v):
        ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                 f'{val:.1f}%', ha='center', fontsize=13, fontweight='bold', color='white')
    ax3.set_ylabel('Taxa de Turnover Anual (%)')
    ax3.set_title('Turnover Empresa\nvs. Benchmark de Mercado')
    ax3.set_ylim(0, 32)
    ax3.text(0.5, 0.06, 'Ref: SHRM, LinkedIn, Glassdoor BR 2024',
             transform=ax3.transAxes, ha='center', fontsize=8, color=MUTED, style='italic')

    plt.tight_layout()
    plt.savefig('08_caged_benchmark.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.show()
else:
    print('Grafico nao gerado: abas do CAGED nao identificadas.')


---
## 📋 Conclusões — Evidências, Discriminação e Plano de Retenção

### 🔎 Evidências Encontradas

| Achado | Evidência | Magnitude |
|--------|-----------|----------|
| Turnover crítico em Vendas | Taxa de 52,3% (2,1× a média) | Custo: R$ 4,1M/ano |
| Gestor tóxico em Vendas | Nota de liderança: 1,13/5 | −59% vs. média geral |
| Gap salarial de gênero | Homens ganham 15–21% mais no mesmo cargo | Confirmado em 100% dos cargos |
| Teto de vidro | Mulheres esperam 38,6% mais para promoção | Mann-Whitney p<0,05 |
| Principal preditor de saída | Satisfação com o trabalho (r=−0,25) | Funcionários com sat.≤2 saem 2× mais |
| Custo total do turnover | 298 desligamentos em 12 meses | R$ 13,4M estimados |

### 🧩 Conexões entre os Datasets (Cruzamentos)

- **Clima + People Analytics:** Correlação negativa entre nota de liderança e taxa de turnover por departamento — piores gestores geram mais saídas.
- **Promoções + People Analytics:** Funcionários sem promoção nos últimos 2 anos têm **10pp a mais** de taxa de saída.
- **Todas as fontes convergem para Vendas:** pior liderança, maior turnover, mais comentários de assédio, menor recomendação.

### 🎯 Plano de Retenção — 3 Horizontes

**🔴 Imediato (0–30 dias)**
1. Afastar gestor de Vendas para investigação de assédio moral — envolver Compliance e Jurídico.
2. Rodar pesquisa de pulso em Vendas para mapear o impacto real.

**🟡 Curto prazo (1–3 meses)**
3. Auditoria salarial completa com revisão por equidade de gênero — corrigir gaps imediatamente.
4. Criar critérios objetivos de promoção com revisão cega de gênero.
5. Ativar o scorecard de risco mensalmente: os ~N funcionários com score > 70 precisam de ação proativa.

**🟢 Médio prazo (3–12 meses)**
6. Programa de desenvolvimento de liderança para todos os gestores.
7. Pesquisas de pulso trimestrais com análise de NLP nos comentários abertos.
8. Meta de paridade salarial por cargo até o final do exercício fiscal.

---
**ROI estimado:** Reduzir turnover de 24,8% → 15% (benchmark setor tech BR) = economia de ~**R$ 5,4M/ano**.  

*Análise: dados de RH internos (1.200 funcionários) + CAGED/MTE (2015–2025).  
Projeto 07 — MBA Ciência de Dados UNIFOR — Prof. Cássio Pinheiro.*